# NOVA 02 — MOD-04 Currency Detection (MobileNetV3-Small)
**Prerequisite:** upload your team-collected CFA images as a **private Kaggle dataset** with the ImageFolder layout (`train|val|test / fcfa_500 ... fcfa_10000`), then attach it here.

No public CFA dataset exists — see section 4.3 of the training guide for the collection protocol (300+ images per denomination per side).

In [ ]:
# ── NOVA bootstrap: clone repo + resolve HF token from Kaggle secret ──
# Before running: Add-ons > Secrets > attach a secret named HF_TOKEN
# (your HuggingFace write token). Settings > Accelerator > GPU T4/P100.
import os, sys, subprocess

REPO = 'https://github.com/BertinAm/nova-ml.git'
if not os.path.exists('/kaggle/working/nova-ml'):
    subprocess.run(['git', 'clone', REPO, '/kaggle/working/nova-ml'], check=True)
os.chdir('/kaggle/working/nova-ml')
sys.path.insert(0, '/kaggle/working/nova-ml/scripts')

from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['NOVA_HF_USERNAME'] = 'unixio'
print('Bootstrap OK — repo cloned, HF token loaded.')

In [ ]:
!pip install -q timm onnx2tf onnx

In [ ]:
# Point at your attached dataset — EDIT the slug to match yours
CFA_DATA = '/kaggle/input/cfa-currency-dataset'   # <-- your dataset slug
!ls {CFA_DATA}/train

In [ ]:
# Two-phase: fine-tune EfficientNet-B4 teacher, distill into MobileNetV3-Small
!python scripts/train_currency_distillation.py --data-dir {CFA_DATA} \
    --teacher-epochs 30 --student-epochs 80 --batch-size 32

In [ ]:
# Held-out test evaluation at the 0.85 confidence gate (FR-04-03)
!python scripts/evaluate_models.py currency \
    --checkpoint /kaggle/working/checkpoints/currency_student_best.pth \
    --data-dir {CFA_DATA}/test

In [ ]:
# Convert to TFLite INT8 (calibrate on val images) + benchmark
!python scripts/convert_to_tflite.py \
    --checkpoint /kaggle/working/checkpoints/currency_student_best.pth \
    --arch mobilenetv3_small_100 --num-classes 5 --input-size 224 \
    --out /kaggle/working/exports/currency_detection_v1.tflite \
    --calib-dir {CFA_DATA}/val --benchmark

In [ ]:
# Publish to HuggingFace: unixio/nova-currency-detection
!python scripts/push_to_huggingface.py --module MOD-04 \
    --pytorch /kaggle/working/checkpoints/currency_student_best.pth \
    --tflite /kaggle/working/exports/currency_detection_v1.tflite \
    --labels labels/cfa_labels.txt \
    --eval-json /kaggle/working/evaluation/currency_test_results.json --version 1.0.0